# Pty-chi Modeling Analysis

This notebook extracts empirical I/O and reconstruction latency coefficients from `pty-chi/modeling_exp/<GPU>/R*/<algorithm>/e*_bs*.log` and the matching power CSV files.

Modeling targets:

- I/O latency: `T_IO(D,G,N) ≈ A_IO(G) + β_IO(G) * R(D) * N^2`
- Reconstruction latency: `T_recon(D,N,E,A,G) ≈ A_recon(A,G) + F_recon(D,N,E,A) / P_eff(A,G)`
- Epoch `1` is treated as warm-up and excluded from reconstruction fitting. Fits use epochs `2, 4, 6`.

Notation:

- `D`: input dataset
- `R(D)`: raw diffraction images, parsed from dataset names like `R8000` or from log data shape
- `N`: diffraction image resolution; current synthetic runs are baseline `N0=64`
- `A`: pty-chi algorithm, e.g. `pie`, `dm`, `lsqml`, `bh`, `ad_ptycho`
- `G`: GPU / accelerator label, e.g. `A100`, `H200`, `MI300X`
- `B`: batch size
- `E`: number of epochs / outer iterations
- `F_recon`: theoretical reconstruction work estimate in GFLOPs
- `P_eff`: fitted effective throughput, not vendor peak TFLOP/s


## FLOPs Model Assumptions

pty-chi algorithms are iterative ptychography engines. The dominant operations are usually:

1. probe/object patch extraction and complex modulation,
2. forward propagation to detector plane, usually a 2D FFT,
3. amplitude / likelihood update against measured diffraction data,
4. backward propagation, usually a 2D IFFT,
5. object/probe/position updates.

For one complex 2D FFT on an `N x N` image, this notebook uses:

`FFT2_flops(N) = 5 * N^2 * (log2(N) + log2(N)) = 10 * N^2 * log2(N)`

This is a standard rough complex FFT count. The total algorithm work per image per epoch is modeled as:

`F_image_epoch(A,N) = q_fft(A) * FFT2_flops(N) + q_elem(A) * N^2`

where `q_fft(A)` is the estimated number of FFT/IFFT-equivalent propagations per image per epoch, and `q_elem(A)` is an approximate count for complex elementwise update work.

Current transparent assumptions:

| algorithm | q_fft | q_elem | rationale |
|---|---:|---:|---|
| `pie` | 2 | 80 | forward FFT + backward IFFT + PIE/ePIE-style updates |
| `dm` | 4 | 120 | difference-map projections use more propagation/update work |
| `lsqml` | 2 | 120 | forward model plus LSQML update/preconditioner-style work |
| `bh` | 2 | 160 | forward model plus extra BH/ADMM-like update state |
| `ad_ptycho` | 4 | 180 | forward pass plus autograd backward roughly doubles propagation work |

Then:

`F_recon(D,N,E,A) = R(D) * E * F_image_epoch(A,N)`

Important: these FLOP constants are an analytical normalization. The fitted `P_eff(A,G)` absorbs modeling error. If we later derive exact kernel-level FLOPs or use profiler FLOPs, replace this table and re-fit.


In [ ]:
from pathlib import Path
import csv
import math
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

cwd = Path.cwd()
if cwd.name == "modeling_exp":
    MODELING_ROOT = cwd
elif (cwd / "modeling_exp").exists():
    MODELING_ROOT = cwd / "modeling_exp"
elif (cwd / "pty-chi" / "modeling_exp").exists():
    MODELING_ROOT = cwd / "pty-chi" / "modeling_exp"
else:
    MODELING_ROOT = Path("/home/zhong.zheng/pty-chi/modeling_exp")

BASELINE_N = 64
FIT_EPOCHS = {2, 4, 6}
ALGORITHM_FLOP_PARAMS = {
    "pie": {"q_fft": 2.0, "q_elem": 80.0},
    "dm": {"q_fft": 4.0, "q_elem": 120.0},
    "lsqml": {"q_fft": 2.0, "q_elem": 120.0},
    "bh": {"q_fft": 2.0, "q_elem": 160.0},
    "ad_ptycho": {"q_fft": 4.0, "q_elem": 180.0},
}
MODELING_ROOT


In [ ]:
LOG_PATTERNS = {
    "data_shape": re.compile(r"data shape:\s*\((\d+),\s*(\d+),\s*(\d+)\)"),
    "io_load_time_s": re.compile(r"^io_load_time_s:\s*([0-9.eE+-]+)", re.MULTILINE),
    "setup_time_s": re.compile(r"^setup_time_s:\s*([0-9.eE+-]+)", re.MULTILINE),
    "task_setup_time_s": re.compile(r"^task_setup_time_s:\s*([0-9.eE+-]+)", re.MULTILINE),
    "reconstruction_run_time_s": re.compile(r"^reconstruction_run_time_s:\s*([0-9.eE+-]+)", re.MULTILINE),
    "save_time_s": re.compile(r"^save_time_s:\s*([0-9.eE+-]+)", re.MULTILINE),
    "total_time_s": re.compile(r"^total_time_s:\s*([0-9.eE+-]+)", re.MULTILINE),
}

def last_float(text, key):
    matches = LOG_PATTERNS[key].findall(text)
    if not matches:
        return np.nan
    return float(matches[-1])

def parse_log(log_path: Path):
    text = log_path.read_text(encoding="utf-8", errors="replace")
    algorithm_dir = log_path.parent
    dataset_dir = algorithm_dir.parent
    gpu_dir = dataset_dir.parent
    gpu = gpu_dir.name
    dataset = dataset_dir.name
    algorithm = algorithm_dir.name

    m_epoch = re.search(r"e(\d+)_bs(\d+)\.log$", log_path.name)
    epochs = int(m_epoch.group(1)) if m_epoch else np.nan
    batch_size = int(m_epoch.group(2)) if m_epoch else np.nan

    shape_match = LOG_PATTERNS["data_shape"].search(text)
    if shape_match:
        raw_images = int(shape_match.group(1))
        n_y = int(shape_match.group(2))
        n_x = int(shape_match.group(3))
    else:
        raw_match = re.fullmatch(r"R(\d+)", dataset)
        raw_images = int(raw_match.group(1)) if raw_match else np.nan
        n_y = n_x = BASELINE_N

    return {
        "gpu": gpu,
        "dataset": dataset,
        "algorithm": algorithm,
        "epochs": epochs,
        "batch_size": batch_size,
        "raw_images": raw_images,
        "resolution_y": n_y,
        "resolution_x": n_x,
        "N": int(n_y) if int(n_y) == int(n_x) else np.nan,
        "io_load_time_s": last_float(text, "io_load_time_s"),
        "setup_time_s": last_float(text, "setup_time_s"),
        "task_setup_time_s": last_float(text, "task_setup_time_s"),
        "reconstruction_run_time_s": last_float(text, "reconstruction_run_time_s"),
        "save_time_s": last_float(text, "save_time_s"),
        "total_time_s": last_float(text, "total_time_s"),
        "log_file": str(log_path),
    }

def parse_power(power_path: Path):
    if not power_path.exists():
        return {"power_csv": str(power_path)}
    rows = []
    with power_path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        power_cols = [c for c in (reader.fieldnames or []) if c.endswith("Power(W)")]
        if not power_cols:
            return {"power_csv": str(power_path)}
        power_col = power_cols[0]
        for row in reader:
            try:
                t = float(row["Time(S)"])
                p = float(row[power_col])
            except (KeyError, TypeError, ValueError):
                continue
            if math.isnan(t) or math.isnan(p):
                continue
            rows.append((t, p))
    if not rows:
        return {"power_csv": str(power_path)}
    times = np.array([r[0] for r in rows], dtype=float)
    powers = np.array([r[1] for r in rows], dtype=float)
    trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return {
        "power_duration_s": float(times[-1]),
        "power_avg_w": float(powers.mean()),
        "power_min_w": float(powers.min()),
        "power_max_w": float(powers.max()),
        "power_energy_j": float(trapz(powers, times)) if len(rows) > 1 else 0.0,
        "power_samples": int(len(rows)),
        "power_csv": str(power_path),
    }

records = []
for log_path in sorted(MODELING_ROOT.glob("*/R*/*/e*_bs*.log")):
    rec = parse_log(log_path)
    rec.update(parse_power(log_path.with_name(log_path.name.replace(".log", "_power.csv"))))
    records.append(rec)

df = pd.DataFrame(records).sort_values(["gpu", "dataset", "algorithm", "epochs", "batch_size"]).reset_index(drop=True)
df["input_pixels"] = df["raw_images"] * df["resolution_x"] * df["resolution_y"]
df["io_stage_s"] = df["io_load_time_s"]
df["prep_stage_s"] = df["setup_time_s"] + df["task_setup_time_s"]
df["pre_recon_s"] = df["io_load_time_s"] + df["setup_time_s"] + df["task_setup_time_s"]
df["num_batches"] = np.ceil(df["raw_images"] / df["batch_size"]).astype("Int64")
df["warmup_epoch"] = df["epochs"] == 1
print(f"Loaded {len(df)} runs from {MODELING_ROOT}")
df.head()


In [ ]:
def fft2_flops(n):
    return 10.0 * n * n * math.log2(n)

def flops_per_image_epoch(algorithm, n):
    params = ALGORITHM_FLOP_PARAMS.get(algorithm)
    if params is None:
        return np.nan
    return params["q_fft"] * fft2_flops(n) + params["q_elem"] * n * n

def total_recon_gflops(row):
    return row["raw_images"] * row["epochs"] * flops_per_image_epoch(row["algorithm"], int(row["N"])) / 1e9

df["flops_per_image_epoch"] = df.apply(lambda r: flops_per_image_epoch(r["algorithm"], int(r["N"])), axis=1)
df["flops_per_image_epoch_gflops"] = df["flops_per_image_epoch"] / 1e9
df["total_recon_gflops"] = df.apply(total_recon_gflops, axis=1)
df["fit_reconstruction"] = df["epochs"].isin(FIT_EPOCHS)

flop_table = pd.DataFrame([
    {
        "algorithm": alg,
        "q_fft": vals["q_fft"],
        "q_elem": vals["q_elem"],
        "F_image_epoch_N64_gflops": flops_per_image_epoch(alg, 64) / 1e9,
        "F_image_epoch_N128_gflops": flops_per_image_epoch(alg, 128) / 1e9,
        "F_image_epoch_N256_gflops": flops_per_image_epoch(alg, 256) / 1e9,
    }
    for alg, vals in ALGORITHM_FLOP_PARAMS.items()
])
flop_table


In [ ]:
def linear_fit(x, y, intercept=True):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) == 0:
        return {"intercept": np.nan, "slope": np.nan, "r2": np.nan, "n": 0}
    if intercept:
        A = np.vstack([np.ones_like(x), x]).T
        coeff, *_ = np.linalg.lstsq(A, y, rcond=None)
        pred = A @ coeff
        a, b = float(coeff[0]), float(coeff[1])
    else:
        denom = float(np.dot(x, x))
        b = float(np.dot(x, y) / denom) if denom else np.nan
        a = 0.0
        pred = b * x
    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {"intercept": a, "slope": b, "r2": r2, "n": int(len(x))}


## I/O Latency Fit

For pty-chi, the measured disk/HDF5 read path is `io_load_time_s`. We fit it as:

`T_IO(D,G,N) ≈ A_IO(G) + β_IO_pixel(G) * R(D) * N^2`

The current runs are all `N=64`, so `β_IO_pixel` is obtained from the slope over `R * N^2`. It should generalize first-order to larger image resolution if the input format and storage stack remain similar.

We also keep an algorithm-specific version because logs are repeated under each algorithm directory, but conceptually the pure HDF5 load should be algorithm independent.


In [ ]:
io_fits = []
for gpu, group in df.groupby("gpu"):
    # Use one representative per GPU/dataset/epoch-independent repeated measurements.
    fit = linear_fit(group["input_pixels"], group["io_stage_s"], intercept=True)
    io_fits.append({
        "gpu": gpu,
        "A_IO_intercept_s": fit["intercept"],
        "beta_IO_s_per_pixel": fit["slope"],
        "beta_IO_ns_per_pixel": fit["slope"] * 1e9,
        "io_fit_r2": fit["r2"],
        "n": fit["n"],
    })
io_coefficients = pd.DataFrame(io_fits).sort_values("gpu")
io_coefficients.round(6)


In [ ]:
io_alg_fits = []
for (gpu, algorithm), group in df.groupby(["gpu", "algorithm"]):
    fit = linear_fit(group["input_pixels"], group["io_stage_s"], intercept=True)
    io_alg_fits.append({
        "gpu": gpu,
        "algorithm": algorithm,
        "A_IO_intercept_s": fit["intercept"],
        "beta_IO_s_per_pixel": fit["slope"],
        "beta_IO_ns_per_pixel": fit["slope"] * 1e9,
        "io_fit_r2": fit["r2"],
        "n": fit["n"],
    })
io_algorithm_coefficients = pd.DataFrame(io_alg_fits).sort_values(["gpu", "algorithm"])
io_algorithm_coefficients.round(6)


## Reconstruction Latency Fit

Epoch `1` is warm-up and is excluded. We use `E = 2, 4, 6` only.

FLOPs-form model:

`T_recon(D,N,E,A,G) ≈ A_recon(A,G) + c(A,G) * F_recon(D,N,E,A)`

where `F_recon` is in GFLOPs, and:

`P_eff(A,G) = 1 / (c(A,G) * 1000)` TFLOP/s

because `c` is seconds/GFLOP.


In [ ]:
fit_df = df[df["fit_reconstruction"]].copy()
recon_fits = []
for (gpu, algorithm, batch_size), group in fit_df.groupby(["gpu", "algorithm", "batch_size"]):
    fit = linear_fit(group["total_recon_gflops"], group["reconstruction_run_time_s"], intercept=True)
    slope = fit["slope"]
    p_eff = 1.0 / (slope * 1000.0) if slope and slope > 0 else np.nan
    recon_fits.append({
        "gpu": gpu,
        "algorithm": algorithm,
        "batch_size": int(batch_size),
        "A_recon_intercept_s": fit["intercept"],
        "s_per_gflop": slope,
        "ms_per_tflop_work": slope * 1e6,
        "P_eff_tflops": p_eff,
        "recon_fit_r2": fit["r2"],
        "n": fit["n"],
    })
reconstruction_flops_coefficients = pd.DataFrame(recon_fits).sort_values(["gpu", "algorithm", "batch_size"])
reconstruction_flops_coefficients.round(6)


In [ ]:
batch_fits = []
for (gpu, algorithm, batch_size), group in fit_df.groupby(["gpu", "algorithm", "batch_size"]):
    work_units = group["epochs"] * np.ceil(group["raw_images"] / group["batch_size"])
    fit = linear_fit(work_units, group["reconstruction_run_time_s"], intercept=True)
    batch_fits.append({
        "gpu": gpu,
        "algorithm": algorithm,
        "batch_size": int(batch_size),
        "A_recon_batch_intercept_s": fit["intercept"],
        "K_s_per_epoch_batch": fit["slope"],
        "K_ms_per_epoch_batch": fit["slope"] * 1e3,
        "batch_epoch_fit_r2": fit["r2"],
        "n": fit["n"],
    })
reconstruction_batch_coefficients = pd.DataFrame(batch_fits).sort_values(["gpu", "algorithm", "batch_size"])
reconstruction_batch_coefficients.round(6)


## Power Windows and Energy Model

The power monitor starts just before I/O. We split each run into coarse windows:

- I/O: `[0, io_load_time_s]`
- setup/task setup: `[io_load_time_s, io_load_time_s + setup_time_s + task_setup_time_s]`
- reconstruction: `[io_load_time_s + setup_time_s + task_setup_time_s, io_load_time_s + setup_time_s + task_setup_time_s + reconstruction_run_time_s]`

For system-flow energy efficiency, the simplest model is:

`E_total = T_IO * P_IO + T_recon * P_recon`

and:

`images/J = R(D) / E_total`


In [ ]:
def power_series(power_csv):
    path = Path(power_csv) if isinstance(power_csv, str) else None
    if path is None or not path.exists():
        return np.array([]), np.array([])
    rows = []
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        power_cols = [c for c in (reader.fieldnames or []) if c.endswith("Power(W)")]
        if not power_cols:
            return np.array([]), np.array([])
        power_col = power_cols[0]
        for row in reader:
            try:
                rows.append((float(row["Time(S)"]), float(row[power_col])))
            except (KeyError, TypeError, ValueError):
                pass
    if not rows:
        return np.array([]), np.array([])
    return np.array([r[0] for r in rows], dtype=float), np.array([r[1] for r in rows], dtype=float)

def window_mean(times, powers, start, end):
    if len(times) == 0 or not np.isfinite(start) or not np.isfinite(end) or end <= start:
        return np.nan
    mask = (times >= start) & (times <= end)
    if not mask.any():
        return np.nan
    return float(powers[mask].mean())

phase_rows = []
for _, row in df.iterrows():
    times, powers = power_series(row.get("power_csv", ""))
    t_io0 = 0.0
    t_io1 = row["io_load_time_s"]
    t_setup1 = row["io_load_time_s"] + row["setup_time_s"] + row["task_setup_time_s"]
    t_recon1 = t_setup1 + row["reconstruction_run_time_s"]
    phase_rows.append({
        "gpu": row["gpu"],
        "dataset": row["dataset"],
        "algorithm": row["algorithm"],
        "epochs": row["epochs"],
        "batch_size": row["batch_size"],
        "raw_images": row["raw_images"],
        "N": row["N"],
        "io_power_w": window_mean(times, powers, t_io0, t_io1),
        "setup_power_w": window_mean(times, powers, t_io1, t_setup1),
        "recon_power_w": window_mean(times, powers, t_setup1, t_recon1),
    })
power_phase_df = pd.DataFrame(phase_rows)
power_phase_df.head()


In [ ]:
power_fits = []
for (gpu, algorithm), group in power_phase_df.dropna(subset=["recon_power_w"]).groupby(["gpu", "algorithm"]):
    fit_group = group[group["epochs"].isin(FIT_EPOCHS)]
    fit = linear_fit(fit_group["epochs"] * np.ceil(fit_group["raw_images"] / fit_group["batch_size"]), fit_group["recon_power_w"], intercept=True)
    power_fits.append({
        "gpu": gpu,
        "algorithm": algorithm,
        "P_IO_w_median": float(group["io_power_w"].median()),
        "P_recon_intercept_w": fit["intercept"],
        "P_recon_slope_w_per_epoch_batch": fit["slope"],
        "P_recon_r2": fit["r2"],
        "n": fit["n"],
    })
power_coefficients = pd.DataFrame(power_fits).sort_values(["gpu", "algorithm"])
power_coefficients.round(4)


In [ ]:
raw_runs_csv = MODELING_ROOT / "ptychi_modeling_runs_extracted.csv"
flop_table_csv = MODELING_ROOT / "ptychi_flops_assumptions.csv"
io_coeff_csv = MODELING_ROOT / "ptychi_io_coefficients.csv"
io_alg_coeff_csv = MODELING_ROOT / "ptychi_io_algorithm_coefficients.csv"
recon_flops_coeff_csv = MODELING_ROOT / "ptychi_reconstruction_flops_coefficients.csv"
recon_batch_coeff_csv = MODELING_ROOT / "ptychi_reconstruction_batch_coefficients.csv"
power_phase_csv = MODELING_ROOT / "ptychi_power_phase_runs.csv"
power_coeff_csv = MODELING_ROOT / "ptychi_power_coefficients.csv"

df.to_csv(raw_runs_csv, index=False)
flop_table.to_csv(flop_table_csv, index=False)
io_coefficients.round(8).to_csv(io_coeff_csv, index=False)
io_algorithm_coefficients.round(8).to_csv(io_alg_coeff_csv, index=False)
reconstruction_flops_coefficients.round(8).to_csv(recon_flops_coeff_csv, index=False)
reconstruction_batch_coefficients.round(8).to_csv(recon_batch_coeff_csv, index=False)
power_phase_df.round(8).to_csv(power_phase_csv, index=False)
power_coefficients.round(8).to_csv(power_coeff_csv, index=False)

print("Saved:")
for path in [raw_runs_csv, flop_table_csv, io_coeff_csv, io_alg_coeff_csv, recon_flops_coeff_csv, recon_batch_coeff_csv, power_phase_csv, power_coeff_csv]:
    print(path)

print("\n=== FLOPs assumptions ===")
print(flop_table.round(6).to_string(index=False))
print("\n=== I/O coefficients ===")
print(io_coefficients.round(6).to_string(index=False))
print("\n=== Reconstruction FLOPs coefficients, fit epochs 2/4/6 ===")
print(reconstruction_flops_coefficients.round(6).to_string(index=False))
print("\n=== Reconstruction batch/epoch coefficients, fit epochs 2/4/6 ===")
print(reconstruction_batch_coefficients.round(6).to_string(index=False))


In [ ]:
print("Pty-chi model formulas")
print("Notation:")
print("  R(D) = raw input diffraction images")
print("  N = image resolution")
print("  E = epochs; epoch=1 is warm-up and excluded from fitting")
print("  A = algorithm")
print("  G = GPU")
print("  B = batch size")
print("  FFT2_flops(N) = 10 * N^2 * log2(N)")
print("  F_image_epoch(A,N) = q_fft(A)*FFT2_flops(N) + q_elem(A)*N^2")
print("  F_recon(D,N,E,A) = R(D) * E * F_image_epoch(A,N) / 1e9 GFLOPs")
print("")

io_rows = {row["gpu"]: row for _, row in io_coefficients.iterrows()}
recon_rows = {(row["gpu"], row["algorithm"]): row for _, row in reconstruction_flops_coefficients.iterrows()}
power_rows = {(row["gpu"], row["algorithm"]): row for _, row in power_coefficients.iterrows()} if len(power_coefficients) else {}

for gpu in sorted(io_coefficients["gpu"].unique()):
    print("=" * 96)
    print(f"GPU: {gpu}")
    io = io_rows[gpu]
    print("I/O latency model:")
    print(f"  T_IO(D,N,{gpu}) = {io['A_IO_intercept_s']:.6f} + {io['beta_IO_ns_per_pixel']:.6f}e-9 * R(D) * N^2 seconds")
    algs = sorted(reconstruction_flops_coefficients.loc[reconstruction_flops_coefficients["gpu"] == gpu, "algorithm"].unique())
    for alg in algs:
        params = ALGORITHM_FLOP_PARAMS[alg]
        recon = recon_rows[(gpu, alg)]
        print(f"  Algorithm: {alg}")
        print(f"    F_image_epoch({alg},N) = {params['q_fft']:.1f} * FFT2_flops(N) + {params['q_elem']:.1f} * N^2 FLOPs")
        print(f"    T_recon(D,N,E,{alg},{gpu}) = {recon['A_recon_intercept_s']:.6f} + {recon['ms_per_tflop_work']:.6f}e-6 * F_recon(D,N,E,{alg}) seconds")
        print(f"    P_eff({alg},{gpu}) = {recon['P_eff_tflops']:.6f} TFLOP/s, R2={recon['recon_fit_r2']:.4f}")
        power = power_rows.get((gpu, alg))
        if power is not None:
            B = int(recon["batch_size"])
            print(f"    P_IO({gpu},{alg}) = {power['P_IO_w_median']:.4f} W")
            print(f"    P_recon(D,E,B={B},{gpu},{alg}) = {power['P_recon_intercept_w']:.4f} + {power['P_recon_slope_w_per_epoch_batch']:.6f} * E * ceil(R(D)/{B}) W")
            print(f"    E_total = T_IO * P_IO + T_recon * P_recon")
            print(f"    images/J = R(D) / E_total")
    print("")
